In [1]:
import torch

In [3]:
%%writefile get_data.py

import requests
import zipfile
from pathlib import Path

data_path = Path("data/")
image_path = data_path / "pizza_stake_sushi"

if image_path.is_dir():
    print(f"directory {image_path} already exists")
else:
    print(f"did not find directory {image_path}, creating...")
    image_path.mkdir(parents = True , exist_ok = True)

with open(data_path / "pizza_stake_sushi.zip","wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza stake sushi data...")
    f.write(request.content)

with zipfile.ZipFile(data_path/"pizza_stake_sushi.zip","r") as zip_ref:
    print(f"Unzipping pizza, steak, suhsi data to {image_path}")
    zip_ref.extractall(image_path)

Writing get_data.py


In [4]:
!python get_data.py

did not find directory data\pizza_stake_sushi, creating...
Unzipping pizza, steak, suhsi data to data\pizza_stake_sushi


In [ ]:
%%writefile data_setup.py

import os

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

NUM_WORKERS = os.cpu_count()

def create_dataloaders(
    train_dir: str,
    test_dir: str,
    transform: transforms.Compose,
    batch_size: int,
    num_workers: NUM_WORKERS
):

train_data = datasets.ImageFolder(train_dir, transform = transform)
test_data = datasets.ImageFolder(test_dir, transform = transform)

class_names = train_data.classes

train_dataloader = DataLoader(
    train_data,
    batch_size = batch_size,
    shuffle= True,
    num_workers = num_workers,
    pin_memory = True
)

test_dataloader = DataLoader(
    test_data,
    batch_size = batch_size,
    shuffle= False,
    num_workers = num_workers,
    pin_memory = True
)

return train_dataloader , test_dataloader, class_names

Writing data_setup.py


In [6]:
%%writefile engine.py

import torch

from tqdm.auto import tqdm
from typing import Dict, List, Tuple

def train_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.Dataloader,
    loss_fn: loss_fn,
    optimizer: torch.optim.Optimizer
    device: torch.device) -> Tuple[float,float]:

    model.train()
    
    train_loss , train_acc = 0,0
    for batch , (X,y) in enumerate(dataloader):
        X,y = X.to("cpu"), y.to("cpu")
    
        y_pred = model(X)
    
        loss = loss_fn(y_pred , y)
        train_loss += loss.item()
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
        y_pred_labels = y_pred.argmax(dim=1)
        train_acc += ((y_pred_labels == y).sum().item()) / len(y_pred)
    
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc 

def test_step(
    model: torch.nn.Module,
    dataloader: torch.utils.data.DataLoader,
    loss_fn: loss_fn,
    device: torch.device) -> Tuple[float,float]:

    model.eval()
    
        test_loss , test_acc = 0,0
    
        with torch.inference_mode():
            for batch , (X,y) in enumerate(dataloader):
                X,y = X.to("cpu"), y.to("cpu")
    
                test_pred_logits = model(X)
    
                loss = loss_fn(test_pred_logits , y)
                test_loss += loss.item()
    
                y_pred_labels = test_pred_logits.argmax(dim=1)
                test_acc += ((y_pred_labels == y).sum().item()) / len(test_pred_logits)
    
        test_loss = test_loss / len(dataloader)
        test_acc = test_acc / len(dataloader)
        return test_loss, test_acc 

def train(
    model: torch.nn.Module,
    train_dataloader: torch.utils.data.DataLoader,
    test_dataloader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    loss_fn: loss_fn,
    epochs: int,
    device: torch.device) -> Dict[str,List]:

    results = {"train_loss": [],
                   "train_acc": [],
                   "test_loss": [],
                   "test_acc": []}
    
        for epoch in tqdm(range(epochs)):
            train_loss , train_acc = train_step(model = model,
                                                dataloader = train_dataloader,
                                                loss_fn = loss_fn,
                                                optimizer = optimizer)
            test_loss , test_acc = test_step(model = model,
                                            dataloader = test_dataloader,
                                            loss_fn = loss_fn)
    
            print(f"Epoch: {epoch+1} | "
                f"train_loss: {train_loss:.4f} | "
                f"train_acc: {train_acc:.4f} | "
                f"test_loss: {test_loss:.4f} | "
                f"test_acc: {test_acc:.4f}"
            )
    
            results['train_loss'].append(train_loss)
            results['train_acc'].append(train_acc)
            results['test_loss'].append(test_loss)
            results['test_acc'].append(test_acc)
    
        return results

Writing engine.py


In [7]:
%%writefile model_builder.py

import torch
from torch import nn

class TinyVGG(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
            super().__init__()
            self.conv_block_1 = nn.Sequential(
                nn.Conv2d(
                in_channels = input_shape,
                out_channels = hidden_units,
                kernel_size = 3,
                stride =1, 
                padding = 1
            ), nn.ReLU(),
                nn.Conv2d(
                in_channels = hidden_units,
                out_channels = hidden_units,
                kernel_size = 3,
                stride =1, 
                padding = 1
            ), nn.ReLU(),
            nn.MaxPool2d(kernel_size = 2)
            )
    
            self.conv_block_2 = nn.Sequential(
                        nn.Conv2d(
                        in_channels = hidden_units,
                        out_channels = hidden_units,
                        kernel_size = 3,
                        stride =1, 
                        padding = 1
                    ), nn.ReLU(),
                        nn.Conv2d(
                        in_channels = hidden_units,
                        out_channels = hidden_units,
                        kernel_size = 3,
                        stride =1, 
                        padding = 1
                    ), nn.ReLU(),
                    nn.MaxPool2d(kernel_size = 2)
                    )
    
            self.classifier = nn.Sequential(
                nn.Flatten(), 
                nn.Linear(in_features = hidden_units* 16 * 16 , out_features = output_shape)
            )
    
        def forward(self , x):
            x = self.conv_block_1(x)
            x = self.conv_block_2(x)
            x = self.classifier(x)
            return x

Writing model_builder.py


In [8]:
%%writefile utils.py

import torch
from pathlib import Path

def save_model(model: torch.nn.Module,
                target_dir: str,
                model_name: str):

            target_dir_path = path(target_dir)
            target_dir_path.mkdir(parents = True, exist_ok= True)

            assert model_name.endswith(".pth") or model_name.endswith(".pt")
            model_save_path = target_dir_path / model_name

            print(f"Saving model to :{model_save_path}")
            torch.save(obj = model.state_dict() , f = model_save_path)


Writing utils.py


In [32]:
%%writefile train.py

import os
import argparse
import torch

from torchvision import transforms

import data_setup, engine, model_builder, utils

parser = argparse.ArgumentParser(description = "Get Some Hyperparameters")

parser.add_argument("--num_epochs",
                        default = 10,
                        type = int,
                        help = "the Number of epochs to train for")

parser.add_argument("--batch_size",
                        type=int,
                        default=32,
                        help="Batch size for training"
)                        

parser.add_argument("--hidden_units",
                        default = 10,
                        type = int,
                        help = "the Number of hidden units in hidden layers")

parser.add_argument("--learning_rate",
                        default = 0.001,
                        type = float,
                        help = "learning rate to use for model")

parser.add_argument("--train_dir",
                        default = "data/pizza_stake_sushi/train",
                        type = str,
                        help = "directory file path to training data in standard image classification format")

parser.add_argument("--test_dir",
                        default = "data/pizza_stake_sushi/test",
                        type = str,
                        help = "directory file path to testing data in standard image classification format")       

args = parser.parse_args()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

NUM_EPOCHS = args.num_epochs
BATCH_SIZE = args.batch_size
HIDDEN_UNITS = args.hidden_units
LEARNING_RATE = args.learning_rate

print(f"[INFO] Training a model for {NUM_EPOCHS} epochs with batch size {BATCH_SIZE} using {HIDDEN_UNITS} hidden units and a learning rate of {LEARNING_RATE}")

train_dir = args.train_dir
test_dir = args.test_dir
print(f"[INFO] Training data file: {train_dir}")
print(f"[INFO] Testing data file: {test_dir}")

data_transform = transforms.Compose([
    transforms.Resize(size = (64,64)),
    transforms.ToTensor()
])

train_dataloader , test_dataloader , class_names = data_setup.create_dataloaders(
    train_dir = train_dir , test_dir = test_dir , transform = data_transform , batch_size = BATCH_SIZE ,num_workers=0
)

model = model_builder.TinyVGG(
    input_shape = 3,
    hidden_units = HIDDEN_UNITS,
    output_shape = len(class_names)
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params = model.parameters() , lr = LEARNING_RATE)

engine.train(model=model,
             train_dataloader=train_dataloader,
             test_dataloader=test_dataloader,
             loss_fn=loss_fn,
             optimizer=optimizer,
             epochs=NUM_EPOCHS,
             device=device)

utils.save_model(model = model,
                target_dir = "models",
                model_name = "05_going_modular_script_mode_tinyvgg_model.pth")

Overwriting train.py


In [35]:
! python train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

[INFO] Using device: cpu
[INFO] Training a model for 5 epochs with batch size 128 using 128 hidden units and a learning rate of 0.0003
[INFO] Training data file: data/pizza_stake_sushi/train
[INFO] Testing data file: data/pizza_stake_sushi/test
Epoch: 1 | train_loss: 1.1055 | train_acc: 0.3214 | test_loss: 1.0951 | test_acc: 0.4133
Epoch: 2 | train_loss: 1.0824 | train_acc: 0.4226 | test_loss: 1.0905 | test_acc: 0.3333
Epoch: 3 | train_loss: 1.0585 | train_acc: 0.3640 | test_loss: 1.0660 | test_acc: 0.4133
Epoch: 4 | train_loss: 1.0043 | train_acc: 0.5971 | test_loss: 1.0137 | test_acc: 0.4267
Epoch: 5 | train_loss: 0.9234 | train_acc: 0.6076 | test_loss: 1.0128 | test_acc: 0.4267
Saving model to :models\05_going_modular_script_mode_tinyvgg_model.pth



  0%|          | 0/5 [00:00<?, ?it/s]c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)

100%|██████████| 5/5 [00:56<00:00, 11.23s/it]


In [42]:
%%writefile predict.py
import torch
import torchvision
from torchvision import transforms
import argparse

import model_builder

parser = argparse.ArgumentParser()

parser.add_argument("--image",
                    help = "target image filepath to predict on")

parser.add_argument("--model_path",
                    default="models/05_going_modular_script_mode_tinyvgg_model.pth",
                    type=str,
                    help="target model to use for prediction filepath")

args = parser.parse_args()
device = "cuda" if torch.cuda.is_available() else "cpu"

class_names = ["pizza" , "steak" , "sushi"]

IMG_PATH = args.image
print(f"[INFO] Predicting on {IMG_PATH}")

def load_model(filepath = args.model_path):
    model = model_builder.TinyVGG(input_shape = 3 , hidden_units = 128, output_shape = 3).to(device)
    print(f"[INFO] Loading in model from: {filepath}")

    model.load_state_dict(torch.load(filepath))
    return model

def pred_image(image_path = IMG_PATH , filepath = args.model_path):
    model = load_model(filepath)
    image = torchvision.io.read_image(str(IMG_PATH)).type(torch.float32)
    image = image/255

    transform = transforms.Resize(size = (64,64))
    image = transform(image)

    model.eval()
    with torch.inference_mode():
        image = image.to(device)

        pred_logits = model(image.unsqueeze(dim=0))
        pred_prob = torch.softmax(pred_logits , dim=1)
        pred_labels = torch.argmax(pred_prob , dim = 1)
        pred_class_name = class_names[pred_labels]

    print(f"[INFO] Pred class: {pred_class_name}, Pred prob: {pred_prob.max():.3f}")

if __name__ == "__main__":
  pred_image()

Overwriting predict.py


In [46]:
! python predict.py --image data/pizza_stake_sushi/test/sushi/175783.jpg

[INFO] Predicting on data/pizza_stake_sushi/test/sushi/175783.jpg
[INFO] Loading in model from: models/05_going_modular_script_mode_tinyvgg_model.pth
[INFO] Pred class: sushi, Pred prob: 0.607
